# **DATA COLLECTION STAGE**
> The `Data` for this research can be collected from the existing literature on **OpenAlex** website. This website is an excellent landing place for scientific research. It's a huge and free scholarly database. 

In [1]:
# importing libraries

import pandas as pd
import requests

In [ ]:
# url = "https://api.openalex.org/works?search=Explainable+AI+Infrastructure&filter=publication_year:2018-2026,type:article"

In [3]:
url = "https://api.openalex.org/works?search=Grad+CAM+Bridge+Inspection&filter=publication_year:2018-2026,type:article"
response = requests.get(url)
data = response.json()

In [4]:
def reconstruct_abstract(inverted_index):
    """
    Convert OpenAlex abstract_inverted_index to readable text.
    """
    if not inverted_index:
        return None

    length = max(max(pos) for pos in inverted_index.values()) + 1
    abstract = [""] * length

    for word, positions in inverted_index.items():
        for pos in positions:
            abstract[pos] = word

    return " ".join(abstract)


def get_authors(authorships):
    """
    Return all author names separated by semicolons.
    """
    if not authorships:
        return None

    return "; ".join(
        author.get("author", {}).get("display_name", "")
        for author in authorships
        if author.get("author")
    )


def get_first_author(authorships):
    """
    Return the first author.
    """
    if not authorships:
        return None

    return authorships[0].get("author", {}).get("display_name")


def get_num_authors(authorships):
    """
    Return number of authors.
    """
    return len(authorships) if authorships else 0


def get_institutions(authorships):
    """
    Return all institutions.
    """
    if not authorships:
        return None

    institutions = set()

    for author in authorships:
        for inst in author.get("institutions", []):
            name = inst.get("display_name")
            if name:
                institutions.add(name)

    return "; ".join(sorted(institutions))


def get_countries(authorships):
    """
    Return all countries represented in the paper.
    """
    if not authorships:
        return None

    countries = set()

    for author in authorships:
        for inst in author.get("institutions", []):
            country = inst.get("country_code")
            if country:
                countries.add(country)

    return "; ".join(sorted(countries))


def get_keywords(keywords):
    """
    Return keywords.
    """
    if not keywords:
        return None

    return "; ".join(
        kw.get("display_name")
        for kw in keywords
        if kw.get("display_name")
    )


def get_num_keywords(keywords):
    return len(keywords) if keywords else 0


def get_concepts(concepts):
    """
    Return concepts.
    """
    if not concepts:
        return None

    return "; ".join(
        concept.get("display_name")
        for concept in concepts
        if concept.get("display_name")
    )


def get_top_concepts(concepts, score_threshold=0.5):
    """
    Return concepts with score above threshold.
    """
    if not concepts:
        return None

    selected = [
        c["display_name"]
        for c in concepts
        if c.get("score", 0) >= score_threshold
    ]

    return "; ".join(selected)


def get_referenced_works(referenced_works):
    """
    Return referenced OpenAlex IDs.
    """
    if not referenced_works:
        return None

    return "; ".join(referenced_works)


def get_journal(paper):
    """
    Return journal name.
    """
    return (
        paper.get("primary_location", {})
        .get("source", {})
        .get("display_name")
    )


def get_publisher(paper):
    """
    Return publisher.
    """
    return (
        paper.get("primary_location", {})
        .get("source", {})
        .get("host_organization_name")
    )


def is_open_access(paper):
    """
    Return Open Access status.
    """
    return paper.get("open_access", {}).get("is_oa")


def get_pdf_url(paper):
    """
    Return PDF URL if available.
    """
    return (
        paper.get("open_access", {})
        .get("oa_url")
    )


def get_landing_page(paper):
    """
    Return landing page URL.
    """
    return (
        paper.get("primary_location", {})
        .get("landing_page_url")
    )


def get_source_type(paper):
    """
    Journal, conference, repository, etc.
    """
    return (
        paper.get("primary_location", {})
        .get("source", {})
        .get("type")
    )

In [5]:
papers = []

for paper in data["results"]:

    papers.append({

        "id": paper.get("id"),
        "doi": paper.get("doi"),

        "title": paper.get("title"),
        "abstract": reconstruct_abstract(
            paper.get("abstract_inverted_index")
        ),

        "publication_year": paper.get("publication_year"),
        "publication_date": paper.get("publication_date"),

        "citation_count": paper.get("cited_by_count"),

        "journal": get_journal(paper),
        "publisher": get_publisher(paper),
        "source_type": get_source_type(paper),

        "authors": get_authors(paper.get("authorships")),
        "first_author": get_first_author(paper.get("authorships")),
        "num_authors": get_num_authors(paper.get("authorships")),

        "institutions": get_institutions(paper.get("authorships")),
        "countries": get_countries(paper.get("authorships")),

        "keywords": get_keywords(paper.get("keywords")),
        "num_keywords": get_num_keywords(paper.get("keywords")),

        "concepts": get_concepts(paper.get("concepts")),
        "top_concepts": get_top_concepts(paper.get("concepts")),

        "referenced_works": get_referenced_works(
            paper.get("referenced_works")
        ),

        "open_access": is_open_access(paper),
        "pdf_url": get_pdf_url(paper),
        "landing_page": get_landing_page(paper),
    })

In [6]:
df = pd.DataFrame(papers)

In [7]:
print("Total number of papers found:", len(df))

Total number of papers found: 25


In [8]:
df.head()

,id,doi,title,abstract,publication_year,publication_date,citation_count,journal,publisher,source_type,...,institutions,countries,keywords,num_keywords,concepts,top_concepts,referenced_works,open_access,pdf_url,landing_page
0,https://openalex.org/W4387722547,https://doi.org/10.1016/j.apenergy.2023.122079,Harnessing eXplainable artificial intelligence...,This study investigates the efficacy of Explai...,2023,2023-10-17,206,Applied Energy,Elsevier BV,journal,...,University of Pretoria,ZA,Feature selection; Time series; Selection (gen...,14,Feature selection; Time series; Selection (gen...,Feature selection; Time series; Selection (gen...,https://openalex.org/W1500895378; https://open...,True,https://doi.org/10.1016/j.apenergy.2023.122079,https://doi.org/10.1016/j.apenergy.2023.122079
1,https://openalex.org/W4407192178,https://doi.org/10.3390/make7010012,Advancing AI Interpretability in Medical Imagi...,This study introduces the Pixel-Level Interpre...,2025,2025-02-06,90,Machine Learning and Knowledge Extraction,Multidisciplinary Digital Publishing Institute,journal,...,Université du Québec à Chicoutimi,CA,Interpretability; Artificial intelligence; Pix...,6,Interpretability; Artificial intelligence; Pix...,Interpretability; Artificial intelligence,https://openalex.org/W1787224781; https://open...,True,https://www.mdpi.com/2504-4990/7/1/12/pdf?vers...,https://doi.org/10.3390/make7010012
2,https://openalex.org/W4403946353,https://doi.org/10.3390/cancers16213668,Grad-CAM Enabled Breast Cancer Classification ...,Breast cancer (BCa) poses a severe threat to w...,2024,2024-10-30,59,Cancers,Multidisciplinary Digital Publishing Institute,journal,...,Higher Institute of Engineering; Horus Univers...,EG; US,Breast cancer; Residual neural network; Medici...,10,Breast cancer; Residual neural network; Medici...,Breast cancer,https://openalex.org/W2149291062; https://open...,True,https://www.mdpi.com/2072-6694/16/21/3668/pdf?...,https://doi.org/10.3390/cancers16213668
3,https://openalex.org/W4386051124,https://doi.org/10.1111/mice.13086,Improving visual question answering for bridge...,This paper explores the application of visual ...,2023,2023-08-22,54,Computer-Aided Civil and Infrastructure Engine...,Wiley,journal,...,RIKEN Center for Advanced Intelligence Project...,JP,Bridge (graph theory); Computer science; Quest...,13,Bridge (graph theory); Computer science; Quest...,Bridge (graph theory); Computer science; Quest...,https://openalex.org/W1514535095; https://open...,True,https://onlinelibrary.wiley.com/doi/pdfdirect/...,https://doi.org/10.1111/mice.13086
4,https://openalex.org/W2981731882,https://doi.org/10.1016/j.inffus.2019.12.012,Explainable Artificial Intelligence (XAI): Con...,None,2019,2019-12-26,9258,Information Fusion,Elsevier BV,journal,...,Institut Systèmes Intelligents et de Robotique...,ES; FR,Computer science; Artificial intelligence; Tax...,11,Computer science; Artificial intelligence; Tax...,Computer science; Artificial intelligence; Tax...,https://openalex.org/W9657784; https://openale...,True,https://arxiv.org/pdf/1910.10045,https://doi.org/10.1016/j.inffus.2019.12.012


In [9]:
df.to_csv('grad_cam_crack_detection.csv', index=False)

load all the csv files collected into a single dataframe

In [2]:
from pathlib import Path
import pandas as pd

## path to the data files
folder_path = Path("../data/raw")

# get all CSV files in the folder
csv_files = folder_path.glob("*.csv")

dfs = [pd.read_csv(file) for file in csv_files]
print(f"Found {len(dfs)} CSV files in the folder '{folder_path}'.")

#merge all dataframes into one
df = pd.concat(dfs, ignore_index=True)

print("Total number of papers found:", len(df))
print("Columns in the dataframe:", df.columns.tolist())

Found 21 CSV files in the folder '..\data\raw'.
Total number of papers found: 525
Columns in the dataframe: ['id', 'doi', 'title', 'abstract', 'publication_year', 'publication_date', 'citation_count', 'journal', 'publisher', 'source_type', 'authors', 'first_author', 'num_authors', 'institutions', 'countries', 'keywords', 'num_keywords', 'concepts', 'top_concepts', 'referenced_works', 'open_access', 'pdf_url', 'landing_page']
